<a href="https://colab.research.google.com/github/MohdZacrySyah/ChatBot-kampus/blob/main/botkampus.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Bersihkan dan pasang versi library yang saling cocok
!pip -q uninstall -y transformers peft sentence-transformers accelerate protobuf >/dev/null
!pip -q install "transformers==4.46.3" "peft==0.13.2" "sentence-transformers==3.0.1" \
                "accelerate>=0.34.2" "python-telegram-bot==21.6" \
                "scikit-learn>=1.3.0" "pandas>=2.0.0" "numpy>=1.24.0" "torch>=2.2.0" \
                "protobuf==5.29.2" "nest-asyncio"

In [ ]:
import os
import re
import logging
import asyncio
import numpy as np
import pandas as pd
import nest_asyncio
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from transformers import pipeline
from telegram import Update, InlineKeyboardButton, InlineKeyboardMarkup
from telegram.ext import (
    Application, CommandHandler, MessageHandler,
    CallbackQueryHandler, filters, ContextTypes
)

In [ ]:
nest_asyncio.apply()

In [ ]:
TOKEN_BOT = "8874276487:AAH40AfmQtcwPCZzP8eL2V1SQ_lUZ0uDSqw"
os.environ["BOT_TOKEN"] = TOKEN_BOT

In [ ]:
# ===== DATASET 90/90/90 =====
data = {
    "Questions": [
        # Akademik - Umum (10)
        "Bagaimana cara mendaftar mata kuliah semester baru?",
        "Kapan jadwal ujian tengah semester?",
        "Dimana lokasi ruang kuliah Gedung A lantai 3?",
        "Bagaimana sistem penilaian di universitas ini?",
        "Apa itu SKS dan bagaimana perhitungannya?",
        "Bagaimana cara mengajukan cuti akademik?",
        "Kapan batas waktu pembayaran UKT?",
        "Bagaimana cara mengecek transkrip nilai?",
        "Apa syarat untuk lulus sarjana?",
        "Bagaimana cara pindah jurusan?",
        # Layanan Mahasiswa (10)
        "Dimana lokasi perpustakaan universitas?",
        "Jam operasional perpustakaan apa saja?",
        "Bagaimana cara meminjam buku di perpustakaan?",
        "Dimana lokasi kantin kampus?",
        "Apakah ada layanan kesehatan di kampus?",
        "Bagaimana cara mengurus kartu mahasiswa yang hilang?",
        "Dimana lokasi ATM terdekat di kampus?",
        "Apakah ada wifi gratis di kampus?",
        "Bagaimana cara akses portal akademik?",
        "Dimana lokasi parkir untuk mahasiswa?",
        # Organisasi dan Kegiatan (10)
        "Apa saja organisasi mahasiswa yang ada?",
        "Bagaimana cara bergabung dengan himpunan mahasiswa?",
        "Kapan jadwal kegiatan orientasi mahasiswa baru?",
        "Apakah ada kegiatan olahraga di kampus?",
        "Bagaimana cara mendaftar lomba yang diadakan kampus?",
        "Apa itu BEM dan fungsinya?",
        "Bagaimana cara mengajukan proposal kegiatan mahasiswa?",
        "Apakah ada program pertukaran mahasiswa?",
        "Bagaimana cara ikut program magang?",
        "Apa saja fasilitas olahraga yang tersedia?",
        # Administrasi (10)
        "Bagaimana cara mengurus surat keterangan mahasiswa aktif?",
        "Dimana lokasi bagian akademik?",
        "Bagaimana cara mengajukan beasiswa?",
        "Apa saja jenis beasiswa yang tersedia?",
        "Bagaimana cara mengurus legalisir ijazah?",
        "Kapan jadwal buka tutup bagian administrasi?",
        "Bagaimana cara mengurus surat pengantar penelitian?",
        "Apa persyaratan untuk wisuda?",
        "Bagaimana cara mengurus surat rekomendasi dosen?",
        "Dimana lokasi bagian kemahasiswaan?",
        # Fasilitas Kampus (10)
        "Apakah ada laboratorium komputer?",
        "Bagaimana cara booking ruang untuk acara?",
        "Dimana lokasi mushola di kampus?",
        "Apakah ada layanan fotokopi di kampus?",
        "Bagaimana cara menggunakan proyektor di kelas?",
        "Dimana lokasi toilet terdekat?",
        "Apakah ada tempat istirahat untuk mahasiswa?",
        "Bagaimana cara mengakses jurnal online?",
        "Dimana lokasi ruang dosen?",
        "Apakah ada kafeteria selain kantin?",
        # Teknologi dan Sistem (10)
        "Bagaimana cara reset password akun mahasiswa?",
        "Kenapa tidak bisa login ke portal akademik?",
        "Bagaimama cara menggunakan aplikasi mobile kampus?",
        "Dimana download aplikasi resmi universitas?",
        "Bagaimana cara mengecek email kampus?",
        "Apa username dan password default sistem?",
        "Bagaimana cara menghubungi IT support?",
        "Kenapa sistem akademik sering error?",
        "Bagaimana cara backup data tugas kuliah?",
        "Apakah bisa akses sistem dari luar kampus?",
        # Tugas Akhir dan Penelitian (10)
        "Bagaimana cara mengajukan judul skripsi?",
        "Siapa saja dosen pembimbing yang tersedia?",
        "Apa persyaratan untuk seminar proposal?",
        "Bagaimana format penulisan skripsi yang benar?",
        "Dimana tempat sidang skripsi dilakukan?",
        "Bagaimana cara mencari referensi untuk penelitian?",
        "Apa itu plagiarisme dan bagaimana menghindarinya?",
        "Bagaimana cara sitasi yang benar?",
        "Kapan jadwal bimbingan dengan dosen?",
        "Bagaimana cara mengurus perizinan penelitian?",
        # Beasiswa (7)
        "Bagaimana cara mempertahankan beasiswa Prestasi?",
        "Apakah penerima KIP Kuliah boleh bekerja paruh waktu?",
        "Kapan pencairan dana beasiswa daerah dilakukan?",
        "Apakah ada bantuan biaya hidup untuk mahasiswa kurang mampu?",
        "Bagaimana jika IPK penerima beasiswa turun di bawah standar?",
        "Apakah boleh mendaftar dua beasiswa sekaligus?",
        "Apa saja komponen berkas berkas untuk pendaftaran beasiswa?",
        # Kelulusan (7)
        "Berapa batas maksimal masa studi untuk program Sarjana?",
        "Apa saja tahapan pendaftaran setelah lulus sidang skripsi?",
        "Bagaimana cara mengecek kuota atau kuota kursi wisuda?",
        "Apakah ada verifikasi ijazah secara nasional?",
        "Bagaimana jika terlambat mengambil ijazah dan transkrip?",
        "Apakah mahasiswa yang lulus tanpa skripsi tetap ikut wisuda?",
        "Dimana tempat pengambilan toga dan atribut wisuda?",
        # Konseling (6)
        "Apakah kampus menyediakan layanan konsultasi psikologi gratis?",
        "Bagaimana cara membuat janji dengan konselor kampus?",
        "Apakah kerahasiaan sesi konseling di kampus terjamin?",
        "Dimana lokasi ruang layanan konseling mahasiswa?",
        "Apakah bisa berkonsultasi mengenai masalah kesulitan belajar?",
        "Apakah layanan konseling terbuka untuk mahasiswa tingkat akhir?",
    ],
    "Answers": [
        # Akademik - Umum (10)
        "Pendaftaran mata kuliah dapat dilakukan melalui portal akademik dengan login menggunakan NIM dan password. Pilih menu KRS, lalu pilih mata kuliah yang ingin diambil sesuai dengan kurikulum program studi Anda.",
        "Jadwal ujian tengah semester biasanya dilaksanakan pada minggu ke-8 semester berjalan. Jadwal detail dapat dilihat di portal akademik atau pengumuman resmi dari bagian akademik.",
        "Ruang kuliah Gedung A lantai 3 terletak di sayap timur kampus. Dari pintu masuk utama, naik tangga atau lift ke lantai 3, kemudian belok ke kanan menuju koridor timur.",
        "Sistem penilaian menggunakan skala A-E dengan bobot: A=4, B=3, C=2, D=1, E=0. Nilai akhir terdiri dari komponen tugas (20%), UTS (30%), UAS (40%), dan partisipasi (10%).",
        "SKS (Sistem Kredit Semester) adalah satuan yang digunakan untuk menyatakan besarnya beban studi mahasiswa. 1 SKS setara dengan 50 menit tatap muka, 60 menit tugas terstruktur, dan 60 menit kegiatan mandiri per minggu.",
        "Pengajuan cuti akademik dilakukan dengan mengisi formulir di bagian akademik, melampirkan surat keterangan dokter (jika sakit) atau alasan lain yang valid, serta mendapat persetujuan dari pembimbing akademik.",
        "Batas waktu pembayaran UKT adalah tanggal 15 setiap bulan untuk semester berjalan. Pembayaran dapat dilakukan melalui bank rekanan atau virtual account yang tertera di tagihan.",
        "Transkrip nilai dapat dicek melalui portal akademik di menu 'Transkrip' atau dengan mengajukan permintaan cetak transkrip resmi di bagian akademik dengan membawa identitas mahasiswa.",
        "Syarat lulus sarjana adalah menyelesaikan minimal 144 SKS dengan IPK minimal 2.0, lulus semua mata kuliah wajib, menyelesaikan skripsi, dan tidak memiliki tunggakan administrasi.",
        "Pindah jurusan dapat dilakukan dengan mengajukan permohonan tertulis ke bagian akademik, memenuhi persyaratan IPK minimal 3.0, dan mengikuti tes seleksi sesuai ketentuan jurusan tujuan.",
        # Layanan Mahasiswa (10)
        "Perpustakaan universitas terletak di Gedung B lantai 1-3, dengan akses utama dari lobby Gedung B. Terdapat juga perpustakaan cabang di beberapa fakultas.",
        "Perpustakaan buka Senin-Jumat pukul 08.00-21.00, Sabtu 08.00-17.00, dan Minggu 13.00-18.00. Pada masa ujian, jam operasional diperpanjang hingga pukul 22.00.",
        "Peminjaman buku dilakukan dengan membawa kartu mahasiswa ke meja sirkulasi. Mahasiswa dapat meminjam maksimal 5 buku selama 7 hari dengan opsi perpanjangan 1x.",
        "Kantin kampus terletak di Gedung C lantai dasar dan food court di lantai 2. Ada juga warung makan di sekitar area parkir belakang kampus.",
        "Ya, tersedia Pusat Kesehatan Mahasiswa (PKM) di Gedung D dengan dokter umum dan perawat. Buka Senin-Jumat 08.00-16.00 dengan layanan konsultasi gratis.",
        "Kartu mahasiswa yang hilang dapat diurus di bagian kemahasiswaan dengan membawa fotokopi KTP, surat keterangan kehilangan dari kepolisian, dan biaya penggantian Rp 50.000.",
        "ATM terdekat tersedia di lobby Gedung A (BCA), dekat kantin (Mandiri), dan di area parkir depan (BRI). Semua ATM beroperasi 24 jam.",
        "Ya, tersedia wifi gratis 'CAMPUS-WIFI' di seluruh area kampus. Login menggunakan NIM dan password yang sama dengan portal akademik.",
        "Portal akademik dapat diakses melalui website resmi universitas di menu 'Portal Mahasiswa' atau langsung ke alamat portal.univ.ac.id dengan login NIM dan password.",
        "Area parkir mahasiswa tersedia di belakang Gedung C dan samping Gedung D. Parkir motor Rp 2.000 dan mobil Rp 5.000 per hari.",
        # Organisasi dan Kegiatan (10)
        "Apa saja organisasi mahasiswa yang ada?",
        "Bergabung dengan himpunan mahasiswa dapat dilakukan saat periode open recruitment yang biasanya diadakan di awal semester. Informasi lengkap tersedia di mading fakultas atau medsos resmi himpunan.",
        "Kegiatan orientasi mahasiswa baru (OSPEK) biasanya dilaksanakan minggu pertama semester ganjil, terdiri dari orientasi universitas, fakultas, dan program studi selama 3-5 hari.",
        "Ya, tersedia berbagai kegiatan olahraga melalui UKM olahraga seperti futsal, basket, voli, badminton, tenis meja, dan cabang olahraga lainnya yang berlatih rutin setiap minggu.",
        "Pendaftaran lomba kampus dapat dilakukan melalui pengumuman resmi di website, media sosial kampus, atau mendaftar langsung ke panitia penyelenggara sesuai persyaratan yang ditentukan.",
        "BEM (Badan Eksekutif Mahasiswa) adalah organisasi eksekutif mahasiswa tertinggi di tingkat universitas yang berfungsi menyalurkan aspirasi mahasiswa dan mengkoordinir kegiatan kemahasiswaan.",
        "Pengajuan proposal kegiatan mahasiswa dilakukan dengan mengisi form proposal, melampirkan RAB dan rundown acara, kemudian diserahkan ke bagian kemahasiswaan untuk mendapat persetujuan.",
        "Ya, tersedia program pertukaran mahasiswa dalam negeri (Merdeka Belajar) dan luar negeri melalui kerjasama dengan universitas partner. Informasi di bagian kerjasama internasional.",
        "Program magang dapat diikuti melalui koordinasi dengan dosen pembimbing atau bagian kerjasama universitas yang memiliki MoU dengan berbagai perusahaan dan instansi.",
        "Fasilitas olahraga meliputi lapangan futsal, basket, voli, tennis, gymnasium, fitness center, kolam renang, dan jogging track yang dapat digunakan mahasiswa dengan jadwal tertentu.",
        # Administrasi (10)
        "Surat keterangan mahasiswa aktif dapat diurus di bagian akademik dengan membawa kartu mahasiswa, mengisi formulir, dan biaya administrasi Rp 10.000. Selesai dalam 1 hari kerja.",
        "Bagian akademik terletak di Gedung A lantai 1, buka Senin-Jumat pukul 08.00-15.00. Untuk pelayanan tertentu tersedia juga saat istirahat dengan sistem piket.",
        "Pengajuan beasiswa dilakukan saat periode pendaftaran dengan melengkapi formulir, transkrip nilai, surat keterangan penghasilan orangtua, dan dokumen pendukung lainnya sesuai jenis beasiswa.",
        "Jenis beasiswa yang tersedia meliputi: Beasiswa Prestasi Akademik, KIP Kuliah, Beasiswa Daerah, Corporate Social Responsibility, dan beasiswa dari yayasan atau lembaga donor.",
        "Legalisir ijazah dapat diurus di bagian akademik dengan membawa ijazah asli dan fotokopi, mengisi formulir legalisir, dan biaya Rp 25.000 per lembar. Proses 2-3 hari kerja.",
        "Bagian administrasi buka Senin-Kamis pukul 08.00-15.00 dan Jumat 08.00-14.30. Pada hari Sabtu buka 08.00-12.00 khusus untuk layanan tertentu.",
        "Surat pengantar penelitian diurus di bagian akademik atau kemahasiswaan dengan melampirkan proposal penelitian, surat dari dosen pembimbing, dan identitas mahasiswa.",
        "Persyaratan wisuda meliputi: lulus semua mata kuliah, IPK minimal 2.0, selesai skripsi dan sidang, tidak ada tunggakan administrasi, dan mengikuti prosesi wisuda sesuai jadwal.",
        "Surat rekomendasi dosen dapat diminta langsung kepada dosen yang bersangkutan dengan memberikan draft surat, CV, transkrip nilai, dan menjelaskan tujuan penggunaan surat tersebut.",
        "Bagian kemahasiswaan terletak di Gedung D lantai 1, menangani urusan organisasi mahasiswa, beasiswa, kegiatan kemahasiswaan, dan layanan konseling mahasiswa.",
        # Fasilitas Kampus (10)
        "Ya, laboratorium komputer tersedia di Gedung B lantai 2 dengan 50 unit PC, buka Senin-Jumat 08.00-20.00 dan Sabtu 08.00-16.00. Dapat digunakan untuk praktikum dan tugas mandiri.",
        "Booking ruang untuk acara dilakukan di bagian umum dengan mengisi formulir peminjaman, melampirkan proposal kegiatan, dan membayar biaya sewa sesuai tarif yang berlaku.",
        "Mushola kampus terletak di Gedung C lantai 2 dengan fasilitas tempat wudhu terpisah putra-putri, sajadah, mukena, dan perpustakaan mini. Buka 24 jam untuk ibadah.",
        "Ya, tersedia layanan fotokopi di beberapa lokasi: dekat perpustakaan, kantin, dan pintu masuk Gedung A dengan tarif Rp 100-200 per lembar tergantung jenis kertas.",
        "Proyektor di kelas dapat digunakan dengan meminjam remote control ke bagian umum atau teknisi dengan meninggalkan kartu identitas sebagai jaminan.",
        "Toilet tersedia di setiap lantai gedung kampus dengan fasilitas terpisah putra-putri. Toilet difabel tersedia di lantai dasar setiap gedang utama.",
        "Ya, tersedia ruang student lounge di setiap gedung dengan fasilitas sofa, meja belajar, dan charging station untuk mahasiswa beristirahat atau belajar kelompok.",
        "Akses jurnal online tersedia melalui perpustakaan digital di website perpustakaan dengan login menggunakan NIM. Tersedia ribuan jurnal nasional dan internasional.",
        "Ruang dosen terletak di setiap gedung fakultas, biasanya di lantai 2-3. Jadwal konsultasi dan office hours dapat dilihat di papan informasi dekat ruang dosen masing-masing.",
        "Ya, selain kantin utama, tersedia kafeteria premium di lantai 3 Gedung A dengan menu lebih bervariasi dan suasana lebih nyaman untuk meeting atau diskusi kelompok.",
        # Teknologi dan Sistem (10)
        "Reset password dapat dilakukan di portal akademik menu 'Lupa Password' dengan memasukkan NIM dan email terdaftar, atau menghubungi bagian IT support dengan membawa kartu mahasiswa.",
        "Masalah login portal akademik dapat disebabkan password salah, koneksi internet, atau maintenance sistem. Coba clear cache browser atau hubungi IT support di ext. 105.",
        "Aplikasi mobile kampus dapat didownload di Play Store/App Store dengan keyword nama universitas. Login menggunakan kredensial yang sama dengan portal akademik.",
        "Download aplikasi resmi universitas tersedia di website resmi bagian download atau scan QR code yang tersedia di poster-poster resmi kampus.",
        "Email kampus dapat diakses melalui webmail di website resmi atau konfigurasi di aplikasi email dengan setting IMAP/POP3 yang disediakan IT support.",
        "Username default adalah NIM mahasiswa, password default biasanya tanggal lahir format DDMMYYYY. Disarankan segera mengganti password setelah login pertama kali.",
        "IT support dapat dihubungi melalui telepon ext. 105, email itsupport@univ.ac.id, atau datang langsung ke ruang IT di Gedung A lantai basement pada jam kerja.",
        "Sistem akademik error dapat disebabkan maintenance rutin, overload traffic, atau masalah teknis. Cek pengumuman resmi atau hubungi IT support untuk konfirmasi status sistem.",
        "Backup data tugas dapat dilakukan dengan menyimpan di cloud storage (Google Drive, OneDrive), external storage, atau email ke diri sendiri untuk antisipasi kehilangan data.",
        "Ya, sistem kampus dapat diakses dari luar kampus melalui koneksi internet. Namun beberapa layanan mungkin dibatasi dan memerlukan VPN untuk akses penuh.",
        # Tugas Akhir dan Penelitian (10)
        "Pengajuan judul skripsi dilakukan dengan mengisi formulir di bagian akademik, melampirkan outline penelitian, dan mendapat persetujuan dari calon dosen pembimbing.",
        "Daftar dosen pembimbing tersedia di bagian akademik atau website fakultas dengan bidang keahlian masing-masing. Mahasiswa dapat memilih sesuai topik penelitian yang diminati.",
        "Persyaratan seminar proposal meliputi: telah menempuh minimal 120 SKS, IPK minimal 2.5, draft proposal telah disetujui pembimbing, dan mendaftar ke bagian akademik.",
        "Format penulisan skripsi mengikuti panduan yang diterbitkan fakultas, umumnya menggunakan font Times New Roman 12pt, spasi 1.5, margin 4-3-3-3 cm, dan sistem referensi sesuai bidang ilmu.",
        "Sidang skripsi dilaksanakan di ruang sidang fakultas atau ruang yang ditentukan bagian akademik. Jadwal sidang disesuaikan dengan ketersediaan dosen penguji.",
        "Pencarian referensi dapat dilakukan melalui perpustakaan digital kampus, Google Scholar, database jurnal online, atau konsultasi dengan pustakawan untuk strategi pencarian yang efektif.",
        "Plagiarisme adalah penggunaan karya orang lain tanpa izin atau pengakuan. Hindari dengan selalu mencantumkan sumber, menggunakan kutipan yang benar, dan paraphrase dengan tepat.",
        "Sitasi yang benar mengikuti style guide yang ditentukan fakultas (APA, Harvard, IEEE, dll). Gunakan reference manager seperti Mendeley atau Zotero untuk memudahkan pengelolaan referensi.",
        "Jadwal bimbingan dapat dikonsultasikan langsung dengan dosen pembimbing. Umumnya dosen memiliki office hours atau dapat membuat janji melalui email atau WhatsApp.",
        "Perizinan penelitian diurus melalui bagian akademik yang akan menerbitkan surat pengantar ke instansi/perusahaan tempat penelitian akan dilakukan.",
        # Beasiswa (7)
        "Untuk mempertahankan beasiswa Prestasi, mahasiswa wajib menjaga IPK minimal 3.25 setiap semester dan tidak terlibat pelanggaran kode etik kampus.",
        "Penerima KIP Kuliah diperbolehkan bekerja paruh waktu asalkan tidak mengganggu aktivitas perkuliahan dan nilai akademik tetap memenuhi standar minimum.",
        "Pencairan dana beasiswa daerah biasanya dilakukan pada bulan ketiga setiap semester baru setelah verifikasi dokumen status mahasiswa aktif selesai.",
        "Ya, kampus menyediakan bantuan biaya hidup dan logistik khusus untuk mahasiswa kurang mampu melalui skema beasiswa internal dan kemitraan yayasan.",
        "Jika IPK turun, penerima beasiswa akan diberikan surat peringatan (SP) dan masa percobaan satu semester untuk memperbaiki nilai sebelum hak beasiswa ditangguhkan.",
        "Mahasiswa tidak diperbolehkan menerima dua beasiswa yang bersumber dari pendanaan APBN/APBD secara bersamaan untuk pemerataan bantuan.",
        "Komponen utama meliputi formulir pendaftaran, transkrip nilai legalisir, surat keterangan tidak mampu (SKTM), slip gaji orang tua, dan surat rekomendasi fakultas.",
        # Kelulusan (7)
        "Batas maksimal masa studi untuk program Sarjana (S1) adalah 14 semester atau 7 tahun akademik berdasarkan regulasi kementerian.",
        "Setelah lulus sidang, mahasiswa wajib melakukan revisi skripsi, mengurus bebas pustaka, mengunggah jurnal ilmiah, dan mendaftar yudisium melalui portal.",
        "Kuota kursi wisuda dapat dipantau secara real-time pada menu pendaftaran wisuda di portal akademik menjelang periode upacara kelulusan dibuka.",
        "Ya, setiap ijazah lulusan akan diverifikasi secara nasional melalui sistem SIVIL (Sistem Verifikasi Ijazah Elektronik) untuk memastikan keasliannya.",
        "Ijazah yang terlambat diambil akan disimpan dengan aman di bagian akademik, namun mahasiswa disarankan segera mengambilnya untuk menghindari biaya penyimpanan tambahan.",
        "Ya, mahasiswa yang menempuh jalur kelulusan non-skripsi (seperti publikasi jurnal bereputasi) tetap diwajibkan mengikuti upacara wisuda resmi.",
        "Pengambilan toga, undangan, dan atribut kelulusan dilakukan di loket bagian kemahasiswaan Gedung D dengan menunjukkan bukti pelunasan biaya wisuda.",
        # Konseling (6)
        "Ya, kampus menyediakan layanan konseling dan psikologi gratis bagi seluruh mahasiswa aktif melalui Unit Pelayanan Konseling di bawah bagian kemahasiswaan.",
        "Membuat janji dapat dilakukan dengan mengisi formulir digital di situs kemahasiswaan atau datang langsung ke ruang administrasi konseling.",
        "Seluruh data pribadi dan isi percakapan selama sesi konseling dijamin kerahasiaannya secara ketat sesuai dengan kode etik psikologi.",
        "Ruang layanan konseling mahasiswa terletak di Gedung D lantai dasar, area belakang dekat taman untuk menjaga kenyamanan dan privasi mahasiswa.",
        "Tentu saja, konselor siap membantu memetakan masalah kesulitan belajar, manajemen waktu, hingga masalah adaptasi lingkungan kampus.",
        "Layanan konseling sangat terbuka untuk mahasiswa tingkat akhir, terutama yang mengalami stres, kecemasan, atau demotivasi dalam menyusun tugas akhir.",
    ],
    "Category": [
        # Akademik - Umum (10)
        "Akademik","Akademik","Akademik","Akademik","Akademik","Akademik","Akademik","Akademik","Akademik","Akademik",
        # Layanan Mahasiswa (10)
        "Layanan","Layanan","Layanan","Layanan","Layanan","Layanan","Layanan","Layanan","Layanan","Layanan",
        # Organisasi dan Kegiatan (10)
        "Organisasi","Organisasi","Organisasi","Organisasi","Organisasi","Organisasi","Organisasi","Organisasi","Organisasi","Organisasi",
        # Administrasi (10)
        "Administrasi","Administrasi","Administrasi","Administrasi","Administrasi","Administrasi","Administrasi","Administrasi","Administrasi","Administrasi",
        # Fasilitas Kampus (10)
        "Fasilitas","Fasilitas","Fasilitas","Fasilitas","Fasilitas","Fasilitas","Fasilitas","Fasilitas","Fasilitas","Fasilitas",
        # Teknologi dan Sistem (10)
        "Teknologi","Teknologi","Teknologi","Teknologi","Teknologi","Teknologi","Teknologi","Teknologi","Teknologi","Teknologi",
        # Tugas Akhir dan Penelitian (10)
        "Penelitian","Penelitian","Penelitian","Penelitian","Penelitian","Penelitian","Penelitian","Penelitian","Penelitian","Penelitian",
        # Beasiswa (7)
        "Beasiswa","Beasiswa","Beasiswa","Beasiswa","Beasiswa","Beasiswa","Beasiswa",
        # Kelulusan (7)
        "Kelulusan","Kelulusan","Kelulusan","Kelulusan","Kelulusan","Kelulusan","Kelulusan",
        # Konseling (6)
        "Konseling","Konseling","Konseling","Konseling","Konseling","Konseling",
    ]
}

In [ ]:
def validate_dataset(d):
    q, a, c = len(d["Questions"]), len(d["Answers"]), len(d["Category"])
    assert q == a == c, f"Panjang tidak sama: Q={q}, A={a}, C={c}"

validate_dataset(data)
df = pd.DataFrame(data)
df.head()

,Questions,Answers,Category
0,Bagaimana cara mendaftar mata kuliah semester ...,Pendaftaran mata kuliah dapat dilakukan melalu...,Akademik
1,Kapan jadwal ujian tengah semester?,Jadwal ujian tengah semester biasanya dilaksan...,Akademik
2,Dimana lokasi ruang kuliah Gedung A lantai 3?,Ruang kuliah Gedung A lantai 3 terletak di say...,Akademik
3,Bagaimana sistem penilaian di universitas ini?,Sistem penilaian menggunakan skala A-E dengan ...,Akademik
4,Apa itu SKS dan bagaimana perhitungannya?,SKS (Sistem Kredit Semester) adalah satuan yan...,Akademik


In [ ]:
def clean_text(text: str) -> str:
    text = text.lower()
    text = re.sub(r"[^\w\s\?\!]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

In [ ]:
sbert_model = SentenceTransformer(
    "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
)

In [ ]:
try:
    sentiment_pipeline = pipeline(
        "sentiment-analysis",
        model="cardiffnlp/twitter-roberta-base-sentiment-latest",
        top_k=None
    )
except Exception as e:
    print(f"Gagal load sentiment: {e}")
    sentiment_pipeline = None

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
question_embeddings = sbert_model.encode(df["Questions"].tolist())
print("Embedding siap")
print("Ukuran embedding:", question_embeddings.shape)

Embedding siap
Ukuran embedding: (90, 384)


In [ ]:
def find_best_answer(user_question: str, threshold: float = 0.30):
    cq = clean_text(user_question)
    user_emb = sbert_model.encode([cq])
    sims = cosine_similarity(user_emb, question_embeddings)[0]
    idx = int(np.argmax(sims))
    score = float(sims[idx])

    if score < threshold:
        return {
            "question": user_question,
            "answer": "Maaf, belum ketemu jawaban yang pas. Coba jelaskan lebih detail.",
            "similarity": score,
            "category": "Unknown",
            "matched_question": None
        }
    return {
        "question": user_question,
        "answer": df.iloc[idx]["Answers"],
        "similarity": score,
        "category": df.iloc[idx]["Category"],
        "matched_question": df.iloc[idx]["Questions"]
    }

In [ ]:
def analyze_sentiment(text: str):
    if sentiment_pipeline is None:
        return {"label": "NEUTRAL", "score": 0.5, "sentiment_score": 0.5}
    out = sentiment_pipeline(text)
    best = max(out[0], key=lambda x: x["score"])
    val = float(best["score"])
    return {
        "label": best["label"].upper(),
        "score": val,
        "sentiment_score": val
    }

In [ ]:
def multi_question_response(user_input: str):
    parts = re.split(
        r'(?:dan\s+(?:bagaimana|apa|dimana|kapan|kenapa|siapa))|(?:\?\s*(?:dan\s*)?(?:bagaimana|apa|dimana|kapan|kenapa|siapa))',
        user_input,
        flags=re.IGNORECASE
    )
    cleaned = []
    for q in parts:
        q = (q or "").strip()
        if len(q) > 5:
            if not q.endswith("?"):
                q += "?"
            cleaned.append(q)

    if not cleaned:
        cleaned = [user_input]

    responses = []
    for q in cleaned[:5]:
        r = find_best_answer(q)
        s = analyze_sentiment(q)
        r.update({
            "sentiment": s["label"],
            "sentiment_score": s["sentiment_score"]
        })
        responses.append(r)
    return responses

In [ ]:
CATEGORY_ORDER = [
    "Akademik",
    "Layanan",
    "Organisasi",
    "Administrasi",
    "Fasilitas",
    "Teknologi",
    "Penelitian",
    "Beasiswa",
    "Kelulusan",
    "Konseling"
]

def get_indices_by_category(cat: str):
    return [i for i, c in enumerate(data["Category"]) if c == cat]

def build_categories_keyboard():
    rows = []
    row = []
    for cat in CATEGORY_ORDER:
        row.append(InlineKeyboardButton(text=cat, callback_data=f"CAT|{cat}"))
        if len(row) == 2:
            rows.append(row)
            row = []
    if row:
        rows.append(row)
    return InlineKeyboardMarkup(rows)

def truncate(txt, maxlen=48):
    return txt if len(txt) <= maxlen else txt[:maxlen - 1] + "…"

def build_questions_keyboard(cat: str):
    idxs = get_indices_by_category(cat)
    rows = []
    for i, idx in enumerate(idxs, 1):
        q = df.iloc[idx]["Questions"]
        rows.append([
            InlineKeyboardButton(
                text=f"{i}. {truncate(q)}",
                callback_data=f"Q|{idx}"
            )
        ])
    rows.append([
        InlineKeyboardButton(text="⬅️ Kategori", callback_data="HOME"),
        InlineKeyboardButton(text="🔁 Pilih Kategori Sama", callback_data=f"CAT|{cat}")
    ])
    return InlineKeyboardMarkup(rows)

In [ ]:
WELCOME = (
    "Halo! Saya Asisten Akademik Kampus.\n"
    "Dibuat oleh Kelompok 4 Text Mining.\n"
    "Pilih kategori di bawah ini, lalu pilih pertanyaannya.\n\n"
    "Anda juga bisa langsung mengetik pertanyaan bebas."
)

async def start(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        WELCOME,
        reply_markup=build_categories_keyboard()
    )

async def help_cmd(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text(
        WELCOME,
        reply_markup=build_categories_keyboard()
    )

async def health(update: Update, context: ContextTypes.DEFAULT_TYPE):
    await update.message.reply_text("OK ✅ — model & embedding aktif.")

In [ ]:
async def on_text(update: Update, context: ContextTypes.DEFAULT_TYPE):
    text = update.message.text or ""
    results = multi_question_response(text)
    blocks = []
    for i, item in enumerate(results, 1):
        sim_pct = f"{item['similarity'] * 100:.1f}%"
        mq = item["matched_question"] or "-"
        blocks.append(
            f"🔹 Pertanyaan {i}\n"
            f"Input: {item['question']}\n"
            f"Kategori: {item['category']} | Kecocokan: {sim_pct}\n"
            f"Padanan: {mq}\n"
            f"Sentimen: {item['sentiment']} ({item['sentiment_score']:.2f})\n"
            f"Jawaban:\n{item['answer']}"
        )
    msg = "\n\n".join(blocks) if blocks else "Maaf, tidak ada pertanyaan yang dipahami."
    await update.message.reply_text(
        msg,
        reply_markup=build_categories_keyboard()
    )

In [ ]:
async def on_callback(update: Update, context: ContextTypes.DEFAULT_TYPE):
    query = update.callback_query
    data_cb = query.data or ""
    await query.answer()

    if data_cb == "HOME":
        await query.edit_message_text(
            "Pilih kategori:",
            reply_markup=build_categories_keyboard()
        )
        return

    if data_cb.startswith("CAT|"):
        cat = data_cb.split("|", 1)[1]
        await query.edit_message_text(
            f"Kategori: *{cat}*\nSilakan pilih pertanyaan:",
            reply_markup=build_questions_keyboard(cat),
            parse_mode="Markdown"
        )
        return

    if data_cb.startswith("Q|"):
        idx = int(data_cb.split("|", 1)[1])
        q = df.iloc[idx]["Questions"]
        a = df.iloc[idx]["Answers"]
        cat = df.iloc[idx]["Category"]
        s = analyze_sentiment(q)
        s_label = s.get("label", "NEUTRAL")
        s_score = s.get("sentiment_score", s.get("score", 0.5))

        text_out = (
            f"*Kategori:* {cat}\n"
            f"*Pertanyaan:* {q}\n"
            f"*Sentimen:* {s_label} ({s_score:.2f})\n\n"
            f"*Jawaban:*\n{a}"
        )
        kb = InlineKeyboardMarkup([
            [InlineKeyboardButton(text="⬅️ Kembali ke Kategori", callback_data=f"CAT|{cat}")],
            [InlineKeyboardButton(text="🏠 Menu Kategori", callback_data="HOME")]
        ])
        await query.edit_message_text(
            text_out,
            parse_mode="Markdown",
            reply_markup=kb
        )
        return

In [ ]:
async def start_bot_notebook():
    token = os.getenv("BOT_TOKEN")
    assert token and token != "MASUKKAN_TOKEN_BOT_ANDA", "Ganti TOKEN_BOT terlebih dahulu."

    app = Application.builder().token(token).concurrent_updates(True).build()

    app.add_handler(CommandHandler("start", start))
    app.add_handler(CommandHandler("help", help_cmd))
    app.add_handler(CommandHandler("health", health))
    app.add_handler(CallbackQueryHandler(on_callback))
    app.add_handler(MessageHandler(filters.TEXT & ~filters.COMMAND, on_text))

    await app.bot.delete_webhook(drop_pending_updates=True)
    await app.initialize()
    await app.start()
    await app.updater.start_polling()

    print("Bot berjalan. Kirim /start di Telegram. Untuk menghentikan, restart runtime.")
    await asyncio.Event().wait()

await start_bot_notebook()

ERROR:telegram.ext.Updater:Exception happened while polling for updates.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_updater.py", line 753, in _network_loop_retry
    if not await do_action():
           ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_updater.py", line 747, in do_action
    return action_cb_task.result()
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/futures.py", line 202, in result
    raise self._exception.with_traceback(self._exception_tb)
  File "/usr/lib/python3.12/asyncio/tasks.py", line 314, in __step_run_and_handle_result
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_updater.py", line 377, in polling_action_cb
    updates = await self.bot.get_updates(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_extbot.py", line 649, in get

Bot berjalan. Kirim /start di Telegram. Untuk menghentikan, restart runtime.


Streaming output truncated to the last 5000 lines.
           ^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_updater.py", line 747, in do_action
    return action_cb_task.result()
           ^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/asyncio/futures.py", line 202, in result
    raise self._exception.with_traceback(self._exception_tb)
  File "/usr/lib/python3.12/asyncio/tasks.py", line 314, in __step_run_and_handle_result
    result = coro.send(None)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_updater.py", line 377, in polling_action_cb
    updates = await self.bot.get_updates(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/ext/_extbot.py", line 649, in get_updates
    updates = await super().get_updates(
              ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/telegram/_bot.py", line 4366, in get_updates
    await s